In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Master_Thesis/data
!pwd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Master_Thesis/data
/content/drive/MyDrive/Master_Thesis/data


In [ ]:
import os
import pandas as pd
import numpy as np
import time
import zipfile

import math
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from typing import Optional

In [ ]:
# -------------------------------------------------------------------
# 1. UNZIP DATA FROM GOOGLE DRIVE TO LOCAL DISK (FAST)
# -------------------------------------------------------------------
zip_path = '/content/drive/MyDrive/Master_Thesis/data/all_data_augmented_text_compressed.zip'
extract_root = '/content/data_unzipped'  # local fast storage

# Only extract if not already extracted (so reruns are instant)
if not os.path.exists(extract_root):
    os.makedirs(extract_root, exist_ok=True)
    print("Extracting ZIP to local disk...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("Extraction complete.")
else:
    print(f"Using already-extracted data at: {extract_root}")

# NOTE: Adjust this depending on how the ZIP is structured.
# Common case: the ZIP contains a folder "all_data_augmented_text/..."
# After extraction, files would be at:
#   /content/data_unzipped/all_data_augmented_text/chorale_0.txt
input_directory = os.path.join(extract_root, 'all_data_augmented_text')

# If your ZIP instead puts the chorale_*.txt files directly under extract_root,
# then use:
# input_directory = extract_root

print("Input directory:", input_directory)

# -------------------------------------------------------------------
# 2. CREATING THE DATA TENSOR
#    Each sequence in the tensor represents a chorale/training block
#    Each sequence is [S, A, T, B, EOS_FLAG]
# -------------------------------------------------------------------
num_chorales = 4584
target_length = 641
vector_length = 5

all_sequences = []

start_time = time.perf_counter()

for i in range(num_chorales):
    filename = f'chorale_{i}.txt'
    filepath = os.path.join(input_directory, filename)

    try:
        with open(filepath, 'r') as f:
            vectors = []
            for line in f:
                numbers = list(map(int, line.strip().split()))
                if len(numbers) != 4:
                    raise ValueError(f"Invalid line in {filename}: {line.strip()}")

                # Explicitly store each number
                soprano, alto, tenor, bass = numbers

                # Create vector of length 5
                vector = [soprano, alto, tenor, bass, 0]

                if len(vector) != vector_length:
                    raise ValueError(
                        f"Vector length mismatch: {len(vector)} instead of {vector_length}"
                    )

                vectors.append(vector)

            # Padding with EOS = 1 at the last index
            current_len = len(vectors)
            if current_len < target_length:
                padding_needed = target_length - current_len
                eos_vector = [0] * (vector_length - 1) + [1]  # Only EOS flag ON
                vectors.extend([eos_vector] * padding_needed)
            elif current_len > target_length:
                raise ValueError(
                    f"Sequence too long in {filename}: length {current_len} > {target_length}"
                )

            # Convert to tensor (shape: [641, 5])
            sequence_tensor = torch.tensor(vectors, dtype=torch.int32)
            all_sequences.append(sequence_tensor)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

end_time = time.perf_counter()
print(f"Loading + padding time (local SSD): {end_time - start_time:.2f} seconds")

# Optionally stack into a single tensor: shape [num_chorales, 641, 5]
data_tensor_all_sequences = torch.stack(all_sequences, dim=0)
print("Final data tensor shape:", data_tensor_all_sequences.shape)

Using already-extracted data at: /content/data_unzipped
Input directory: /content/data_unzipped/all_data_augmented_text
Loading + padding time (local SSD): 3.49 seconds
Final data tensor shape: torch.Size([4584, 641, 5])


In [ ]:
print(len(all_sequences))
data_tensor_all_sequences = torch.stack(all_sequences).int()
print(data_tensor_all_sequences.shape)
print(data_tensor_all_sequences)

4584
torch.Size([4584, 641, 5])
tensor([[[74, 70, 65, 58,  0],
         [74, 70, 65, 58,  0],
         [74, 70, 65, 58,  0],
         ...,
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1]],

        [[69, 64, 61, 57,  0],
         [69, 64, 61, 57,  0],
         [69, 64, 61, 57,  0],
         ...,
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1]],

        [[67, 62, 59, 55,  0],
         [67, 62, 59, 55,  0],
         [67, 62, 59, 55,  0],
         ...,
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1]],

        ...,

        [[59, 54, 51, 47,  0],
         [59, 54, 51, 47,  0],
         [59, 54, 51, 47,  0],
         ...,
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1],
         [ 0,  0,  0,  0,  1]],

        [[58, 53, 50, 46,  0],
         [58, 53, 50, 46,  0],
         [58, 53, 50, 46,  0],
         ...,
         [ 0,  0,  0,  0,  1],
        

In [ ]:
                               ┌────────────────────────────┐
Data: X ∈ ℝ[B, T, 5]           │ X[:, 0:640] = inputs       │
B = batch size                 │ X[:, 1:641] = targets      │
T = 641                        └────────────────────────────┘

                            ┌──────────────────────────────────────────────┐
                            │                  INPUT TOKEN                 │
                            │              x ∈ ℤ^{B×T×5} (raw)             │
                            │     [S_midi, A_midi, T_midi, B_midi, EOS]    │
                            └──────────────────────────────────────────────┘
                                             │
                                             ▼
        ┌────────────────────────────────────────────────────────────────────┐
        │                 REST-AWARE TOKENIZATION + EMBEDDING                │
        │  MIDI → class indices per voice:                                   │
        │    REST=0, NOTE=(midi - MIN + 1)                                   │
        │  Embeddings per time step:                                         │
        │    e_S,e_A,e_T,e_B ∈ ℝ^{64},  e_EOS ∈ ℝ^{16}                       │
        │  Concat: [e_S;e_A;e_T;e_B;e_EOS] ∈ ℝ^{272}                         │
        │  Linear projection: 272 → 384                                      │
        │  Output: H ∈ ℝ^{B×T×384}                                           │
        └────────────────────────────────────────────────────────────────────┘
                          │                                   │
                          │                                   │
                          ▼                                   ▼
┌──────────────────────────────────────────────┐   ┌──────────────────────────────────────────────┐
│            TRANSFORMER BRANCH (GLOBAL)       │   │             TEMPORAL CNN BRANCH (LOCAL)      │
│         8 × TransformerBlock (Pre-LN)        │   │            5 × CausalConvBlock (TCN)         │
│                                              │   │                                              │
│  Each block:                                 │   │  Each block:                                 │
│   LN → RoPE Causal Self-Attn → Residual      │   │   Conv1d (causal) → GELU                     │
│   LN → MLP(384→1536→384) → Dropout → Residual│   │   Conv1d (causal) → GELU → Dropout → Residual│
│                                              │   │  Dilations (if enabled): 1,2,4,8,16          │
│  Causal mask: prevents attending to future   │   │  Kernel size: 7                              │
│  RoPE: rotates Q,K with relative position info│  │                                              │
│                                              │   │  Final LayerNorm                             │
│  Final LayerNorm                             │   │  Output: H_cnn ∈ ℝ^{B×T×384}                 │
│  Output: H_trf ∈ ℝ^{B×T×384}                 │   └──────────────────────────────────────────────┘
└──────────────────────────────────────────────┘
                          │                                   │
                          └───────────────┬───────────────────┘
                                          │
                                          ▼
                     ┌──────────────────────────────────────────────┐
                     │                    FUSION                    │
                     │   Concat: [H_trf ; H_cnn] ∈ ℝ^{B×T×768}      │
                     │   Linear: 768 → 384                          │
                     │   GELU                                       │
                     │   Output: H_fused ∈ ℝ^{B×T×384}              │
                     └──────────────────────────────────────────────┘
                                          │
                                          ▼
                     ┌──────────────────────────────────────────────┐
                     │         POST-FUSION REFINEMENT (SHARED)      │
                     │   MLP: 384 → 768 → 384                       │
                     │   Residual: H_fused + MLP(H_fused)           │
                     │   LayerNorm                                  │
                     │   Output: H_refined ∈ ℝ^{B×T×384}            │
                     └──────────────────────────────────────────────┘
                                          │
                                          ▼
┌────────────────────────────────────────────────────────────────────────────┐
│                              OUTPUT HEADS (PER t)                          │
│                                                                            │
│   S head:   Linear 384 → P_S=34   → logits_S   ∈ ℝ^{B×T×34}                │
│   A head:   Linear 384 → P_A=34   → logits_A   ∈ ℝ^{B×T×34}                │
│   T head:   Linear 384 → P_T=36   → logits_T   ∈ ℝ^{B×T×36}                │
│   B head:   Linear 384 → P_B=43   → logits_B   ∈ ℝ^{B×T×43}                │
│   EOS head: Linear 384 → 1        → logits_EOS ∈ ℝ^{B×T×1}                 │
└────────────────────────────────────────────────────────────────────────────┘


In [ ]:

# ---------------------------------------------------------
# Configuration dataclass for easy hyperparameter tweaking
# ---------------------------------------------------------

@dataclass
class HybridConfig:
    # Core model dims
    d_model: int = 384       # main hidden size
    d_pitch: int = 64        # per-voice pitch embedding size
    d_eos: int = 16          # EOS embedding size

    # Transformer
    n_layers: int = 8
    n_heads: int = 8
    mlp_ratio: float = 4.0
    attn_dropout: float = 0.0
    resid_dropout: float = 0.1

    # CNN
    cnn_layers: int = 5
    cnn_kernel_size: int = 7
    cnn_dropout: float = 0.1
    cnn_dilated: bool = True  # use dilations 1,2,4,...

    # Positional embeddings (RoPE)
    rope_base: int = 10000


# ---------------------------------------------------------
# MIDI ranges for each voice, used for remapping and sizes
# ---------------------------------------------------------

S_MIN, S_MAX = 49, 81   # soprano
A_MIN, A_MAX = 42, 74   # alto
T_MIN, T_MAX = 35, 69   # tenor
B_MIN, B_MAX = 25, 66   # bass

# +1 for REST (class 0) for each voice
P_S = (S_MAX - S_MIN + 1) + 1 # number of classes for S
P_A = (A_MAX - A_MIN + 1) + 1 # number of classes for A
P_T = (T_MAX - T_MIN + 1) + 1 # number of classes for T
P_B = (B_MAX - B_MIN + 1) + 1 # number of classes for B


# ---------------------------------------------------------
# Rotary positional embeddings (RoPE) utilities
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    """
    Computes rotary positional embeddings for a given head dimension.

    This module produces cos/sin tables used to rotate Q and K in attention,
    encoding relative position information without explicit additive PE.
    """
    def __init__(self, dim: int, base: int = 10000):
        super().__init__()
        assert dim % 2 == 0, "RoPE dimension must be even"
        self.dim = dim
        # inverse frequencies for different dimensions
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len: int, device=None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns cos and sin tables for positions 0..seq_len-1.

        Shapes:
          cos, sin: [seq_len, dim]
        """
        device = device if device is not None else self.inv_freq.device
        # positions 0,1,...,seq_len-1
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        # outer product -> [seq_len, dim/2]
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        # duplicate for even/odd dimensions
        emb = torch.cat([freqs, freqs], dim=-1)  # [seq_len, dim]
        cos = emb.cos()
        sin = emb.sin()
        return cos, sin


def apply_rotary(q: torch.Tensor,
                 k: torch.Tensor,
                 cos: torch.Tensor,
                 sin: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Apply rotary embeddings to queries and keys.

    q, k: [B, n_heads, T, head_dim]
    cos, sin: [T, head_dim]

    We broadcast cos/sin to [1,1,T,head_dim] and apply rotation on pairs of dims.
    """
    # reshape cos/sin for broadcasting
    cos = cos[None, None, :, :]  # [1,1,T,head_dim]
    sin = sin[None, None, :, :]  # [1,1,T,head_dim]

    # split into even and odd dimensions
    q_even, q_odd = q[..., 0::2], q[..., 1::2]
    k_even, k_odd = k[..., 0::2], k[..., 1::2]

    # rotate
    q_rot_even = q_even * cos[..., 0::2] - q_odd * sin[..., 0::2]
    q_rot_odd  = q_even * sin[..., 0::2] + q_odd * cos[..., 0::2]
    k_rot_even = k_even * cos[..., 0::2] - k_odd * sin[..., 0::2]
    k_rot_odd  = k_even * sin[..., 0::2] + k_odd * cos[..., 0::2]

    # interleave even/odd back together
    q_rot = torch.empty_like(q)
    k_rot = torch.empty_like(k)
    q_rot[..., 0::2] = q_rot_even
    q_rot[..., 1::2] = q_rot_odd
    k_rot[..., 0::2] = k_rot_even
    k_rot[..., 1::2] = k_rot_odd

    return q_rot, k_rot


# ---------------------------------------------------------
# Causal self-attention layer with RoPE
# ---------------------------------------------------------

class CausalSelfAttention(nn.Module):
    """
    Multi-head self-attention with:
      - Causal masking (no attention to future tokens)
      - Rotary positional embeddings applied to Q, K
    """
    def __init__(self, d_model: int, n_heads: int, rope_base: int = 10000,
                 attn_dropout: float = 0.0, resid_dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"

        # Linear layer that produces Q, K, V at once: [B,T,d] -> [B,T,3d]
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        # Output projection after attention
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.resid_dropout = nn.Dropout(resid_dropout)

        # Rotary embedding for head_dim
        self.rope = RotaryEmbedding(self.head_dim, base=rope_base)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, d_model]
        attn_mask: [1, 1, T, T] with 0 on allowed positions and -inf on masked positions
        """
        B, T, C = x.shape

        # Project to QKV and split
        qkv = self.qkv(x)  # [B, T, 3*d_model]
        q, k, v = qkv.chunk(3, dim=-1)  # each [B, T, d_model]

        # Reshape for multi-head: [B, T, n_heads, head_dim] -> [B, n_heads, T, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Get RoPE cos/sin tables and apply to Q, K
        cos, sin = self.rope(seq_len=T, device=x.device)  # [T, head_dim]
        q, k = apply_rotary(q, k, cos, sin)

        # Scaled dot-product attention with causal mask
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [B,nH,T,T]
        attn_scores = attn_scores + attn_mask  # add -inf to masked positions

        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)

        # attention * values
        y = attn_probs @ v  # [B, nH, T, head_dim]

        # Merge heads back: [B, T, d_model]
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y


# ---------------------------------------------------------
# Transformer block (LN + RoPE attention + MLP + residual)
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    """
    Standard Transformer block but:
      - Uses our CausalSelfAttention (with RoPE)
      - Uses LayerNorm before attention and MLP (Pre-LN)
    """
    def __init__(self, d_model: int, n_heads: int,
                 mlp_ratio: float = 4.0,
                 attn_dropout: float = 0.0,
                 resid_dropout: float = 0.1,
                 rope_base: int = 10000):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            rope_base=rope_base,
            attn_dropout=attn_dropout,
            resid_dropout=resid_dropout,
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, int(mlp_ratio * d_model)),
            nn.GELU(),
            nn.Linear(int(mlp_ratio * d_model), d_model),
            nn.Dropout(resid_dropout),
        )

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        # Self-attention with pre-LN and residual
        x = x + self.attn(self.ln1(x), attn_mask=attn_mask)
        # Feed-forward network with pre-LN and residual
        x = x + self.mlp(self.ln2(x))
        return x


# ---------------------------------------------------------
# Temporal CNN (causal, TCN-style)
# ---------------------------------------------------------

class CausalConvBlock(nn.Module):
    """
    A single causal convolutional block (like a TCN block):
      - Conv1d (causal via left padding) + GELU
      - Conv1d (causal) + GELU
      - Dropout
      - Residual connection
    """
    def __init__(self, channels: int, kernel_size: int = 5,
                 dilation: int = 1, dropout: float = 0.1):
        super().__init__()
        # Padding to maintain causality (only depends on past)
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            channels, channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.conv2 = nn.Conv1d(
            channels, channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.padding = padding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, C, T]
        We use left padding and then cut off the extra right side to ensure causality.
        """
        residual = x
        y = self.conv1(x)
        if self.padding > 0:
            y = y[..., :-self.padding]
        y = self.act(y)

        y = self.conv2(y)
        if self.padding > 0:
            y = y[..., :-self.padding]
        y = self.act(y)
        y = self.dropout(y)

        return residual + y


class TemporalCNN(nn.Module):
    """
    Stack of causal convolutional blocks operating over time dimension.

    Input:  [B, T, d_model]
    Output: [B, T, d_model]  (same shape)
    """
    def __init__(self, d_model: int, n_layers: int = 3,
                 kernel_size: int = 5, dropout: float = 0.1,
                 dilated: bool = True):
        super().__init__()
        blocks = []
        for i in range(n_layers):
            dilation = 2 ** i if dilated else 1
            blocks.append(CausalConvBlock(
                channels=d_model,
                kernel_size=kernel_size,
                dilation=dilation,
                dropout=dropout,
            ))
        self.blocks = nn.ModuleList(blocks)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, d_model] -> transpose to [B, d_model, T] for Conv1d,
           run through TCN blocks, then transpose back.
        """
        x_c = x.transpose(1, 2)  # [B, d_model, T]
        for block in self.blocks:
            x_c = block(x_c)
        x_out = x_c.transpose(1, 2)  # [B, T, d_model]
        return self.ln(x_out)


# ---------------------------------------------------------
# Main hybrid model: embeddings + Transformer + CNN + heads
# ---------------------------------------------------------

class HybridChoraleModel(nn.Module):
    """
    Hybrid Transformer+CNN model for 4-voice chorale next-step prediction.

    Expected input:
      x: [B, T, 5]
         x[..., 0] = S (soprano MIDI in [49,81])
         x[..., 1] = A (alto MIDI in [42,74])
         x[..., 2] = T (tenor MIDI in [35,69])
         x[..., 3] = B (bass  MIDI in [25,66])
         x[..., 4] = EOS flag in {0,1}

    Output:
      dict with logits:
        "S":   [B, T, P_S]  (33 classes)
        "A":   [B, T, P_A]  (33 classes)
        "T":   [B, T, P_T]  (35 classes)
        "B":   [B, T, P_B]  (42 classes)
        "EOS": [B, T, 1]    (logit for EOS)
    """
    def __init__(self, config: HybridConfig):
        super().__init__()
        self.config = config

        # ---- Pitch embeddings per voice (using class indices) ----
        self.emb_S = nn.Embedding(P_S, config.d_pitch)
        self.emb_A = nn.Embedding(P_A, config.d_pitch)
        self.emb_T = nn.Embedding(P_T, config.d_pitch)
        self.emb_B = nn.Embedding(P_B, config.d_pitch)

        # ---- EOS embedding (0 or 1) ----
        self.emb_EOS = nn.Embedding(2, config.d_eos)

        # Total input embedding size before projection
        input_embed_dim = 4 * config.d_pitch + config.d_eos

        # ---- Project concatenated embeddings to model dimension ----
        self.input_proj = nn.Linear(input_embed_dim, config.d_model)

        # ---- Transformer stack ----
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.d_model,
                n_heads=config.n_heads,
                mlp_ratio=config.mlp_ratio,
                attn_dropout=config.attn_dropout,
                resid_dropout=config.resid_dropout,
                rope_base=config.rope_base,
            )
            for _ in range(config.n_layers)
        ])
        self.ln_trf = nn.LayerNorm(config.d_model)

        # ---- Temporal CNN branch ----
        self.cnn = TemporalCNN(
            d_model=config.d_model,
            n_layers=config.cnn_layers,
            kernel_size=config.cnn_kernel_size,
            dropout=config.cnn_dropout,
            dilated=config.cnn_dilated,
        )

        # ---- Fusion (concat transformer and cnn outputs, then project) ----
        self.fuse = nn.Linear(2 * config.d_model, config.d_model)
        self.fuse_act = nn.GELU()

        # ---- Shared post-fusion MLP (refines combined representation) ----
        # Tweak mid_dim for more/less capacity.
        mid_dim = 2 * config.d_model  # e.g. 2x bottleneck
        self.post_fuse_mlp = nn.Sequential(
            nn.Linear(config.d_model, mid_dim),
            nn.GELU(),
            nn.Linear(mid_dim, config.d_model),
        )
        self.post_fuse_ln = nn.LayerNorm(config.d_model)

        # ---- Output heads: separate logits for each voice and EOS ----
        self.head_S = nn.Linear(config.d_model, P_S)   # 33 classes for S
        self.head_A = nn.Linear(config.d_model, P_A)   # 33 classes for A
        self.head_T = nn.Linear(config.d_model, P_T)   # 35 classes for T
        self.head_B = nn.Linear(config.d_model, P_B)   # 42 classes for B
        self.head_EOS = nn.Linear(config.d_model, 1)   # binary EOS logit

    @staticmethod
    def _build_causal_mask(T: int, device) -> torch.Tensor:
        """
        Build a [1,1,T,T] mask with 0 for allowed positions and -inf for masked.

        Used to ensure each position only attends to itself and previous positions.
        """
        mask = torch.full((T, T), float("-inf"), device=device)
        mask = torch.triu(mask, diagonal=1)  # above main diagonal -> -inf
        return mask.unsqueeze(0).unsqueeze(0)  # [1,1,T,T]

    def _embed_inputs(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, 5] with raw MIDI/EOS.
          - MIDI 0 means REST for that voice.
          - MIDI > 0 is a pitched note in the allowed range.
        Returns:
          H: [B, T, d_model] embedded sequence.
        """
        # Split out each component
        S_midi = x[..., 0]  # [B, T]
        A_midi = x[..., 1]
        T_midi = x[..., 2]
        B_midi = x[..., 3]
        EOS = x[..., 4].long()  # ensure integer 0/1

        # REST masks: True where midi == 0
        S_rest = (S_midi == 0)
        A_rest = (A_midi == 0)
        T_rest = (T_midi == 0)
        B_rest = (B_midi == 0)

        # Initialize class indices with 0 = REST
        S_cls = torch.zeros_like(S_midi, dtype=torch.long)
        A_cls = torch.zeros_like(A_midi, dtype=torch.long)
        T_cls = torch.zeros_like(T_midi, dtype=torch.long)
        B_cls = torch.zeros_like(B_midi, dtype=torch.long)

        # For non-rest positions, map MIDI → class in [1..P_voice-1]
        S_cls[~S_rest] = (S_midi[~S_rest] - S_MIN + 1).long()
        A_cls[~A_rest] = (A_midi[~A_rest] - A_MIN + 1).long()
        T_cls[~T_rest] = (T_midi[~T_rest] - T_MIN + 1).long()
        B_cls[~B_rest] = (B_midi[~B_rest] - B_MIN + 1).long()

        # Safety checks
        # assert torch.all((S_cls >= 0) & (S_cls < P_S)), "Soprano MIDI out of range"
        # assert torch.all((A_cls >= 0) & (A_cls < P_A)), "Alto MIDI out of range"
        # assert torch.all((T_cls >= 0) & (T_cls < P_T)), "Tenor MIDI out of range"
        # assert torch.all((B_cls >= 0) & (B_cls < P_B)), "Bass MIDI out of range"

        # Voice-specific pitch embeddings
        e_S = self.emb_S(S_cls)  # [B, T, d_pitch]
        e_A = self.emb_A(A_cls)
        e_T = self.emb_T(T_cls)
        e_B = self.emb_B(B_cls)

        # EOS embedding
        e_EOS = self.emb_EOS(EOS)  # [B, T, d_eos]

        # Concatenate all embeddings along feature dimension
        # Result: [B, T, 4*d_pitch + d_eos]
        concat_embed = torch.cat([e_S, e_A, e_T, e_B, e_EOS], dim=-1)

        # Project to model dimension: [B, T, d_model]
        H = self.input_proj(concat_embed)
        return H


    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass of the model.

        x: [B, T, 5] tensor of raw MIDI/EOS (as described above).
        Returns:
          {
            "S":   logits for soprano classes [B, T, P_S],
            "A":   logits for alto classes    [B, T, P_A],
            "T":   logits for tenor classes   [B, T, P_T],
            "B":   logits for bass classes    [B, T, P_B],
            "EOS": logits for EOS (binary)    [B, T, 1],
          }
        """
        B, T, F = x.shape
        assert F == 5, "Input last dimension must be 5: [S,A,T,B,EOS]"

        # ---- 1) Embed inputs to [B, T, d_model] ----
        H = self._embed_inputs(x)

        # ---- 2) Transformer branch (with causal mask + RoPE) ----
        attn_mask = self._build_causal_mask(T, device=x.device)
        H_trf = H
        for blk in self.blocks:
            H_trf = blk(H_trf, attn_mask=attn_mask)
        H_trf = self.ln_trf(H_trf)  # [B, T, d_model]

        # ---- 3) CNN branch ----
        H_cnn = self.cnn(H)  # [B, T, d_model]

        # ---- 4) Fusion ----
        H_cat = torch.cat([H_trf, H_cnn], dim=-1)   # [B, T, 2*d_model]
        H_fused = self.fuse_act(self.fuse(H_cat))   # [B, T, d_model]

        # ---- 4b) Shared post-fusion MLP + residual + LN ----
        H_mlp = self.post_fuse_mlp(H_fused)              # [B, T, d_model]
        H_refined = self.post_fuse_ln(H_fused + H_mlp)   # [B, T, d_model]

        # ---- 5) Output heads for each voice and EOS ---- V1 uses H_fused below, to include 4b) use H_refined
        logits_S = self.head_S(H_refined)            # [B, T, P_S]
        logits_A = self.head_A(H_refined)            # [B, T, P_A]
        logits_T = self.head_T(H_refined)            # [B, T, P_T]
        logits_B = self.head_B(H_refined)            # [B, T, P_B]
        logits_EOS = self.head_EOS(H_refined)        # [B, T, 1]

        return {
            "S": logits_S,
            "A": logits_A,
            "T": logits_T,
            "B": logits_B,
            "EOS": logits_EOS,
        }

In [ ]:
# ---------------------------------------------------------
# Dataset wrapper
# ---------------------------------------------------------

class ChoraleDataset(Dataset):
    """
    Wraps your full data tensor of shape [num_sequences, 641, 5]
    so that each item is a full sequence (a training block).

    Each __getitem__ returns one sequence: [641, 5].
    """
    def __init__(self, data_tensor: torch.Tensor):
        """
        data_tensor: torch.Tensor of shape [N_seq, 641, 5]
        """
        super().__init__()
        assert data_tensor.ndim == 3
        assert data_tensor.size(1) == 641
        assert data_tensor.size(2) == 5
        self.data = data_tensor

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        # Return a single sequence [641,5]
        return self.data[idx]


# ---------------------------------------------------------
# Loss computation helper
# ---------------------------------------------------------

def compute_loss(model: nn.Module, batch: torch.Tensor) -> torch.Tensor:
    """
    Compute next-step prediction loss on a batch of full sequences.

    batch: [B, 641, 5] with raw MIDI/EOS.
    We do language-model style teacher forcing:
      - Inputs:  x_in  = batch[:, :-1, :]  (0..639)
      - Targets: y_out = batch[:, 1:,  :]  (1..640)
    Loss:
      - CE over S/A/T/B on every time step
      - BCEWithLogits over EOS on every time step
    """
    device = next(model.parameters()).device

    # Move batch to the same device as model
    batch = batch.to(device)  # [B, 641, 5]
    B, T_full, num_dim_vector = batch.shape
    assert T_full == 641 and num_dim_vector == 5

    # Input and target sequences (LM-style shift)
    x_in = batch[:, :-1, :]   # [B, 640, 5] - condition on chords 0..639
    y_out = batch[:, 1:, :]   # [B, 640, 5] - predict chords 1..640

    # Forward pass: model predicts logits for each time step in x_in
    outputs = model(x_in)
    logits_S = outputs["S"]   # [B, 640, P_S]
    logits_A = outputs["A"]   # [B, 640, P_A]
    logits_T = outputs["T"]   # [B, 640, P_T]
    logits_B = outputs["B"]   # [B, 640, P_B]
    logits_EOS = outputs["EOS"]  # [B, 640, 1]

    # -------------------------------------------------
    # Convert y_out MIDI to class indices per voice
    # -------------------------------------------------

    S_midi = y_out[..., 0]  # [B, 640]
    A_midi = y_out[..., 1]
    T_midi = y_out[..., 2]
    B_midi = y_out[..., 3]
    EOS    = y_out[..., 4].float()  # [B, 640]

    # REST masks: True where midi == 0
    S_rest = (S_midi == 0)
    A_rest = (A_midi == 0)
    T_rest = (T_midi == 0)
    B_rest = (B_midi == 0)

    # Initialize class indices with 0 = REST
    S_cls = torch.zeros_like(S_midi, dtype=torch.long)
    A_cls = torch.zeros_like(A_midi, dtype=torch.long)
    T_cls = torch.zeros_like(T_midi, dtype=torch.long)
    B_cls = torch.zeros_like(B_midi, dtype=torch.long)

    # For non-rest positions, map MIDI → class in [1..P_voice-1]
    S_cls[~S_rest] = (S_midi[~S_rest] - S_MIN + 1).long()
    A_cls[~A_rest] = (A_midi[~A_rest] - A_MIN + 1).long()
    T_cls[~T_rest] = (T_midi[~T_rest] - T_MIN + 1).long()
    B_cls[~B_rest] = (B_midi[~B_rest] - B_MIN + 1).long()

    # Flatten batch/time dims for CE losses
    B_T = B * (T_full - 1)  # B * 640

    loss_S = F.cross_entropy(
        logits_S.reshape(B_T, P_S),  # [B*640, P_S]
        S_cls.reshape(B_T)           # [B*640]
    )
    loss_A = F.cross_entropy(
        logits_A.reshape(B_T, P_A),
        A_cls.reshape(B_T)
    )
    loss_T = F.cross_entropy(
        logits_T.reshape(B_T, P_T),
        T_cls.reshape(B_T)
    )
    loss_B = F.cross_entropy(
        logits_B.reshape(B_T, P_B),
        B_cls.reshape(B_T)
    )

    loss_EOS = F.binary_cross_entropy_with_logits(
        logits_EOS.squeeze(-1),  # [B, 640]
        EOS                      # [B, 640]
    )

    # Total loss (you can add weights if needed)
    loss = loss_S + loss_A + loss_T + loss_B + loss_EOS
    return loss


# ---------------------------------------------------------
# Training loop with train/val split + EARLY STOPPING
# ---------------------------------------------------------

def train_model(
    data_tensor: torch.Tensor,
    num_epochs: int = 10,
    batch_size: int = 16,
    learning_rate: float = 3e-4,
    weight_decay: float = 0.01,
    grad_clip: float = 1.0,
    val_fraction: float = 0.1,
    seed: int = 42,
    patience: int = 4,                   #early stopping patience
    save_path: str = "best_model_holder.pt",    #where to save best model
):
    """
    Train the HybridChoraleModel with a train/validation split.

    data_tensor: [N_seq, 641, 5] torch.Tensor with raw MIDI/EOS.
    val_fraction: fraction of sequences to hold out for validation (e.g. 0.1 for 10%).
    patience: how many epochs without val improvement before early stopping.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Reproducible split
    g = torch.Generator().manual_seed(seed)

    # ---- Create dataset and split into train/val ----
    full_dataset = ChoraleDataset(data_tensor)
    n_total = len(full_dataset)
    n_val = int(n_total * val_fraction)
    n_train = n_total - n_val

    train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val], generator=g)

    print(f"Total sequences: {n_total}")
    print(f"Train sequences: {n_train}")
    print(f"Val sequences:   {n_val}")

    # ---- DataLoaders ----
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    # ---- Initialize model and optimizer ----
    config = HybridConfig()
    model = HybridChoraleModel(config).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # For recording history if you want to plot later
    train_losses = []
    val_losses = []

    # ---- EARLY STOPPING tracking (NEW) ----
    best_val_loss = float("inf")   # <-- NEW: best validation loss so far
    best_epoch = -1                # <-- NEW: which epoch it occurred
    patience_counter = 0           # <-- NEW: how many epochs since last improvement

    # ---- Training loop ----
    for epoch in range(1, num_epochs + 1):
        # -----------------------
        # Training phase
        # -----------------------
        model.train()
        running_train_loss = 0.0
        num_train_batches = 0

        for batch in train_loader:
            optimizer.zero_grad()
            loss = compute_loss(model, batch)
            loss.backward()

            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            running_train_loss += loss.item()
            num_train_batches += 1

        avg_train_loss = running_train_loss / max(1, num_train_batches)
        train_losses.append(avg_train_loss)

        # -----------------------
        # Validation phase
        # -----------------------
        model.eval()
        running_val_loss = 0.0
        num_val_batches = 0

        with torch.no_grad():
            for batch in val_loader:
                loss = compute_loss(model, batch)
                running_val_loss += loss.item()
                num_val_batches += 1

        avg_val_loss = running_val_loss / max(1, num_val_batches)
        val_losses.append(avg_val_loss)

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

        # -----------------------
        # EARLY STOPPING logic
        # -----------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            patience_counter = 0

            # Save best model weights
            torch.save(model.state_dict(), save_path)
            print(f"  -> New best val loss! Saving model at epoch {epoch}.")
        else:
            patience_counter += 1
            print(f"  -> No improvement for {patience_counter} epoch(s).")

            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch}")
                break

    # ---- Load best model before returning ----
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"Loaded best model from epoch {best_epoch} with val loss {best_val_loss:.4f}")

    return model, train_losses, val_losses

In [ ]:

# data_tensor: torch.Tensor of shape [4584, 641, 5]
trained_model, train_losses, val_losses = train_model(
    data_tensor=data_tensor_all_sequences,
    num_epochs=20,
    batch_size=6,
    learning_rate=3e-4,
    val_fraction=0.1,   # 10% of sequences for validation
)

#save with:
%cd /content/drive/MyDrive/Master_Thesis/Saved_Models
torch.save(trained_model.state_dict(), "hybrid_chorale_model_improvedv2_20epochs_earlyStop_6batch_3e-4learningRate_01valFraction.pt")


Using device: cuda
Total sequences: 4584
Train sequences: 4126
Val sequences:   458
Epoch 001 | Train Loss: 1.1768 | Val Loss: 0.8172
  -> New best val loss! Saving model at epoch 1.
Epoch 002 | Train Loss: 0.7508 | Val Loss: 0.7292
  -> New best val loss! Saving model at epoch 2.
Epoch 003 | Train Loss: 0.6377 | Val Loss: 0.6974
  -> New best val loss! Saving model at epoch 3.
Epoch 004 | Train Loss: 0.5306 | Val Loss: 0.7006
  -> No improvement for 1 epoch(s).
Epoch 005 | Train Loss: 0.4217 | Val Loss: 0.7329
  -> No improvement for 2 epoch(s).
Epoch 006 | Train Loss: 0.3257 | Val Loss: 0.8099
  -> No improvement for 3 epoch(s).
Epoch 007 | Train Loss: 0.2509 | Val Loss: 0.8690
  -> No improvement for 4 epoch(s).
Early stopping triggered at epoch 7. Best epoch: 3
Loaded best model from epoch 3 with val loss 0.6974
/content/drive/MyDrive/Master_Thesis/Saved_Models


In [ ]:

@torch.no_grad()
def generate_chorale(
    model: nn.Module,
    start_sequence: torch.Tensor,
    max_steps: int = 640,
    eos_threshold: float = 0.5,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    sample: bool = False,
    device: Optional[torch.device] = None,
) -> torch.Tensor:
    """
    Autoregressively generate a 4-voice chorale using the trained HybridChoraleModel.

    Args:
        model:          Trained HybridChoraleModel.
        start_sequence: Tensor of shape [T0, 5] or [1, T0, 5] with raw MIDI/EOS:
                          [..., 0] = S in [49,81]
                          [..., 1] = A in [42,74]
                          [..., 2] = T in [35,69]
                          [..., 3] = B in [25,66]
                          [..., 4] = EOS ∈ {0,1}
                        Typically EOS=0 for all initial chords.
        max_steps:      Maximum number of new chords to generate (in addition to start_sequence).
                        Note: generation stops early if EOS is predicted.
        eos_threshold:  Threshold on sigmoid(EOS_logit) to decide EOS=1 (stop).
        temperature:    Temperature for sampling; 1.0 = no change, >1 = smoother, <1 = sharper.
        top_k:          If not None and sample=True, restrict sampling to top_k probabilities.
        sample:         If False → greedy decoding (argmax). If True → stochastic sampling.
        device:         Optional torch.device to move model & data to. Defaults to model's device.

    Returns:
        A tensor of shape [T_generated, 5] on CPU with raw MIDI/EOS values, where:
          - T_generated = start_sequence_length + number_of_generated_steps
    """
    # -------------------------------------------------
    # Setup: device, eval mode, initial sequence
    # -------------------------------------------------
    model.eval()

    if device is None:
        device = next(model.parameters()).device

    # Ensure we have a batch dimension of 1: [1, T0, 5]
    if start_sequence.ndim == 2:
        # [T0, 5] -> [1, T0, 5]
        seq = start_sequence.unsqueeze(0).to(device)
    elif start_sequence.ndim == 3 and start_sequence.size(0) == 1:
        seq = start_sequence.to(device)
    else:
        raise ValueError("start_sequence must be [T0, 5] or [1, T0, 5]")

    # Make sure EOS is float or int (0/1)
    seq[..., 4] = seq[..., 4].round()

    # -------------------------------------------------
    # Helper: top-k filtering for logits (optional)
    # -------------------------------------------------
    def top_k_filter(logits, k):
        """
        Keep only top-k logits, set the rest to -inf so they never get sampled.
        logits: [num_classes]
        """
        if k is None or k <= 0 or k >= logits.size(-1):
            return logits
        # Find the cutoff logit
        values, _ = torch.topk(logits, k)
        min_topk = values[..., -1, None]  # scalar threshold
        # Mask out everything below that
        return torch.where(logits < min_topk, torch.full_like(logits, float('-inf')), logits)

    # -------------------------------------------------
    # Main autoregressive loop
    # -------------------------------------------------
    # We will generate up to 'max_steps' new chords.
    for step in range(max_steps):
        B, T_current, vector_size = seq.shape
        assert B == 1 and vector_size == 5

        # Forward pass through the model with current prefix
        outputs = model(seq)  # logits for all positions: shapes [1, T_current, ...]
        logits_S = outputs["S"]   # [1, T_current, P_S]
        logits_A = outputs["A"]   # [1, T_current, P_A]
        logits_T = outputs["T"]   # [1, T_current, P_T]
        logits_B = outputs["B"]   # [1, T_current, P_B]
        logits_EOS = outputs["EOS"]  # [1, T_current, 1]

        # We only care about the LAST time step's logits: position T_current-1
        last_logits_S = logits_S[0, -1, :]   # [P_S]
        last_logits_A = logits_A[0, -1, :]   # [P_A]
        last_logits_T = logits_T[0, -1, :]   # [P_T]
        last_logits_B = logits_B[0, -1, :]   # [P_B]
        last_logit_EOS = logits_EOS[0, -1, 0]  # scalar

        # -------------------------------------------------
        # Turn logits into next pitch class predictions
        # -------------------------------------------------
        def choose_pitch(logits, n_classes):
            """
            Given logits [n_classes], choose the next class index.
            - If sample=False: greedy argmax.
            - If sample=True: softmax sampling with temperature and top-k.
            """
            # temperature scaling
            if temperature != 1.0:
                logits = logits / temperature

            if sample:
                # optional top-k filtering
                logits = top_k_filter(logits, top_k)
                probs = F.softmax(logits, dim=-1)
                # sample one index from the categorical distribution
                idx = torch.multinomial(probs, num_samples=1).item()
            else:
                # greedy
                idx = torch.argmax(logits).item()
            return idx

        # Choose class indices for each voice
        S_idx = choose_pitch(last_logits_S, P_S)
        A_idx = choose_pitch(last_logits_A, P_A)
        T_idx = choose_pitch(last_logits_T, P_T)
        B_idx = choose_pitch(last_logits_B, P_B)

        # Helper: convert class index back to MIDI, with 0 = REST
        def class_to_midi(idx, vmin):
            if idx == 0:
                return 0  # REST
            else:
                return (idx - 1) + vmin

        S_midi = class_to_midi(S_idx, S_MIN)
        A_midi = class_to_midi(A_idx, A_MIN)
        T_midi = class_to_midi(T_idx, T_MIN)
        B_midi = class_to_midi(B_idx, B_MIN)

        # -------------------------------------------------
        # Decide EOS
        # -------------------------------------------------
        # Convert EOS logit to probability via sigmoid
        eos_prob = torch.sigmoid(last_logit_EOS).item()

        eos_flag = 1 if eos_prob > eos_threshold else 0

        # -------------------------------------------------
        # Append new chord to sequence
        # -------------------------------------------------
        new_chord = torch.tensor(
            [[S_midi, A_midi, T_midi, B_midi, eos_flag]],
            dtype=seq.dtype,
            device=device,
        ).unsqueeze(0)  # shape [1, 1, 5]

        seq = torch.cat([seq, new_chord], dim=1)  # [1, T_current+1, 5]

        # -------------------------------------------------
        # Stopping condition if EOS was predicted
        # -------------------------------------------------
        if eos_flag == 1.0:
            break

    # Return generated sequence without batch dimension, moved to CPU
    return seq.squeeze(0).to("cpu")

In [ ]:
#Assuming trained model and below:
config = HybridConfig()
model = HybridChoraleModel(config)
model.load_state_dict(torch.load("hybrid_chorale_model_improvedv2_20epochs_earlyStop_6batch_3e-4learningRate_01valFraction.pt", map_location="cpu"))
model.to("cuda")  # if available

HybridChoraleModel(
  (emb_S): Embedding(34, 64)
  (emb_A): Embedding(34, 64)
  (emb_T): Embedding(36, 64)
  (emb_B): Embedding(43, 64)
  (emb_EOS): Embedding(2, 16)
  (input_proj): Linear(in_features=272, out_features=384, bias=True)
  (blocks): ModuleList(
    (0-7): 8 x TransformerBlock(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=False)
        (out_proj): Linear(in_features=384, out_features=384, bias=False)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
        (rope): RotaryEmbedding()
      )
      (ln2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1536, out_features=384, bias=True)
        (3): Dropout(p=0.1, inplace=False)
    

In [ ]:
#Prepare a starting chord or prefix:
# Example: one starting chord (no EOS yet)

start_seq = torch.tensor([
    [64, 60, 55, 48, 0] # [S, A, T, B, EOS]
], dtype=torch.float32)  # shape [1,5] → will become [1, T0=1, 5]


#Generate:

generated = generate_chorale(
    model,
    start_sequence=start_seq,
    max_steps=640,      # maximum additional chords
    eos_threshold=0.5,  # when EOS prob > 0.5, stop
    temperature=0.3,
    top_k=5,
    sample=True,       # greedy decoding or not
)

print(generated.shape)  # [T_gen, 5]
print(generated[:10])   # first 10 chords

torch.Size([641, 5])
tensor([[64., 60., 55., 48.,  0.],
        [64., 60., 55., 48.,  0.],
        [64., 60., 55., 48.,  0.],
        [64., 60., 55., 48.,  0.],
        [67., 60., 53., 45.,  0.],
        [67., 60., 53., 45.,  0.],
        [67., 60., 53., 45.,  0.],
        [67., 60., 53., 45.,  0.],
        [67., 59., 53., 52.,  0.],
        [67., 59., 53., 52.,  0.]])


In [ ]:
output_path = '/content/drive/MyDrive/Master_Thesis/Generated_Music/Generated_by_improved_v2/generated_chorale_12.txt'

with open(output_path, "w") as f:
    for row in generated:
        # take first 4 entries, convert to ints, join as strings
        line = " ".join(str(int(x.item())) for x in row[:4])
        f.write(line + "\n")

print(f"Saved to {output_path}")

Saved to /content/drive/MyDrive/Master_Thesis/Generated_Music/Generated_by_improved_v2/generated_chorale_12.txt


In [ ]:
#SOME START SEQUENCES:

start_seq = torch.tensor([
    [63, 58, 55, 51, 0],
    [63, 58, 55, 51, 0],
    [63, 58, 55, 51, 0],
    [63, 58, 55, 51, 0],
    [63, 60, 56, 56, 0],
    [63, 60, 56, 56, 0],
    [63, 60, 56, 55, 0],
    [63, 60, 56, 55, 0],
    [65, 60, 56, 53, 0],
    [65, 60, 56, 59, 0],
    [65, 59, 56, 53, 0],
    [65, 60, 56, 53, 0] # [S, A, T, B, EOS]
], dtype=torch.float32)


#[63, 58, 55, 51, 0]

#[67, 62, 59, 55, 0]

#[72, 65, 57, 53, 0]

#[65, 62, 58, 46, 0]

#[66, 62, 57, 50, 0]





